# Resume RAG Experimentation

This notebook builds the index, runs sample job matching, and records retrieval metrics.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from resume_rag import build_resume_index, match_job_description, evaluate_retrieval

RESUME_ROOT = ROOT / 'data' / 'resumes'
INDEX_DIR = ROOT / 'data' / 'index'
JOB_DIR = ROOT / 'data' / 'jobs'

ModuleNotFoundError: No module named 'resume_rag'

In [ ]:
jobs = [
    (JOB_DIR / 'data_scientist.txt').read_text(encoding='utf-8'),
    (JOB_DIR / 'ml_engineer.txt').read_text(encoding='utf-8'),
    (JOB_DIR / 'data_engineer.txt').read_text(encoding='utf-8'),
    (JOB_DIR / 'product_analyst.txt').read_text(encoding='utf-8'),
    (JOB_DIR / 'full_stack_engineer.txt').read_text(encoding='utf-8'),
]
targets = ['Aisha Khan', 'Marcus Lee', 'Nina Patel', 'Olivia Chen', 'Sophie Martin']
metrics = evaluate_retrieval(jobs, targets, storage_dir=INDEX_DIR)
metrics

## Observations

- The model performs best when the job description and resume share explicit skill terms, especially Python, SQL, and role-specific tooling.
- The remaining miss is likely driven by overlap in generic analytics language across multiple resumes, which makes the top ranking less decisive.
- Latency is low enough for interactive experimentation on this small corpus, so the main improvement area is ranking quality rather than runtime.

## Results Summary

| Metric | Value | Notes |
| --- | --- | --- |
| Retrieval accuracy | 0.80 | 4 out of 5 sample jobs returned the expected top candidate |
| Avg latency | 51.0 ms | Local TF-IDF backend on the sample corpus |
| P95 latency | 73.6 ms | Still comfortably under 100 ms in this run |

## Retrieval Metrics

The following cell checks top-1 retrieval accuracy on the included sample jobs and measures latency.

In [ ]:
store = build_resume_index(resume_root=RESUME_ROOT, storage_dir=INDEX_DIR)
len(store.records)

In [ ]:
job_description = (JOB_DIR / 'data_scientist.txt').read_text(encoding='utf-8')
result = match_job_description(job_description, storage_dir=INDEX_DIR, top_k=5)
json.dumps(result, indent=2)[:4000]